## 1)Imports & Setup

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cpu"
CSV_PATH = "base_features_with_image.csv"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2)Load & Clean Data

In [2]:
df = pd.read_csv(CSV_PATH)

df['description'] = df['description'].fillna("")
df['msrp'] = pd.to_numeric(df['msrp'], errors='coerce').fillna(df['msrp'].median())
df['clip_image_text_sim'] = pd.to_numeric(df['clip_image_text_sim'], errors='coerce').fillna(0.0)
df['clip_image_text_sim'] = ((df['clip_image_text_sim'] + 1) / 2).clip(0, 1)

df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0.0)
df['price_anomaly_z'] = pd.to_numeric(df['price_anomaly_z'], errors='coerce').fillna(0.0)

# If no labels exist, create them
if 'label_suspicious' not in df.columns:
    def auto_label(r):
        r1 = r['price_anomaly_z'] < -1.5
        r2 = r['clip_image_text_sim'] < 0.25
        r3 = (r['price'] < 0.35 * r['msrp']) if r['msrp'] > 0 else False
        return 1 if (r1 or r2 or r3) else 0

    df['label_suspicious'] = df.apply(auto_label, axis=1)

print("Dataset loaded:", df.shape)
print(df['label_suspicious'].value_counts())

Dataset loaded: (29339, 9)
label_suspicious
1    23048
0     6291
Name: count, dtype: int64


## 3)Build Unified Text Field & Generate MiniLM Embeddings

In [3]:
df['text_full'] = (
    df['title'].astype(str) + " " +
    df['description'].astype(str) + " " +
    df['category'].astype(str)
).str[:1000]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts = df['text_full'].tolist()

embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/459 [00:00<?, ?it/s]

Embedding shape: (29339, 384)


## 4)PCA Reduction + Normalize Numeric Features

In [4]:
pca = PCA(n_components=64, random_state=SEED)
emb_pca = pca.fit_transform(embeddings)

num_cols = ["price", "msrp", "price_anomaly_z", "clip_image_text_sim"]

scaler = StandardScaler()
X_num = scaler.fit_transform(df[num_cols])

X = np.hstack([emb_pca, X_num]).astype(np.float32)
y = df["label_suspicious"].astype(np.float32).values

print("Final X shape:", X.shape)

Final X shape: (29339, 68)


## 5)Train/Validation Split + DataLoaders

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

## 6)MLP Model Definition

In [6]:
class MLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256), 
            nn.ReLU(), 
            nn.Dropout(0.2),
            nn.Linear(256, 64), 
            nn.ReLU(), 
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model = MLP(X.shape[1])
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

## 7)Training Loop With Best Model Saving

In [7]:
from sklearn.metrics import average_precision_score

best_ap = -1

for epoch in range(12):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    # Eval
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            probs = torch.sigmoid(model(xb)).numpy()
            preds.extend(probs)
            trues.extend(yb.numpy())

    ap = average_precision_score(trues, preds)
    print(f"Epoch {epoch+1} | AP: {ap:.4f}")

    if ap > best_ap:
        best_ap = ap
        torch.save(model.state_dict(), "suspicion_model.pt")
        pickle.dump(pca, open("pca_transform.pkl", "wb"))
        pickle.dump(scaler, open("scaler.pkl", "wb"))
        print("Saved best model!")

print("Best AP:", best_ap)

Epoch 1 | AP: 0.9991
Saved best model!
Epoch 2 | AP: 0.9996
Saved best model!
Epoch 3 | AP: 0.9997
Saved best model!
Epoch 4 | AP: 0.9996
Epoch 5 | AP: 0.9998
Saved best model!
Epoch 6 | AP: 0.9999
Saved best model!
Epoch 7 | AP: 0.9998
Epoch 8 | AP: 0.9998
Epoch 9 | AP: 0.9998
Epoch 10 | AP: 0.9998
Epoch 11 | AP: 0.9998
Epoch 12 | AP: 0.9998
Best AP: 0.9998613917683263
